## Gossiping
Avevamo introdotto il gossiping cercando di risolvere il problema della propagazione delle informazioni tra nodi, insieme al flooding e la random walk.  
Con il gossiping in particolare ci si chiede **come diffondere un messaggio senza assegnare al nodo iniziale tutto il lavoro di invio --> l'idea è che anche i nodi che ricevono l'aggiornamento contribuiscano a propagarlo**.

Il **multicast** è un modello di comunicazione dove un messaggio è destinato a un **gruppo di ricevitori**: può essere *uno-a-molti* (es. distribuzione video da sorgente a più utenti) o *molti-a-molti* (es. videoconferenza).  
Il broadcast rappresenta un caso particolare di multicast dove i destinatari sono tutti i nodi della rete.

Per capire il problema di scalabilità, immaginiamo che un nodo $A$ debba mandare lo stesso msg a $B$ e $C$ e che i percorsi verso $B$ e $C$ condividano un link --> se $A$ effettua due invii *unicast* --> su quel link viaggeranno due copie, quando ne bastava una sola...  
Il principio per ridurre la msg complexity è quindi **far viaggiare una sola copia sul link comune e duplicarla alla diramazione**.

Il multicast si può realizzare:
- *a livello di rete* (IP multicast): i router replicano i pacchetti dove i percorsi si diramano, verso le destinazioni appropriate. Tuttavia non tutti i router supportano il multicast..
- **a livello applicativo**: i nodi dell'applicazione sappiamo infatti che scambiano messaggi in una **rete overlay** su cui potremmo realizzare il multicast!

Nell'**overlay strutturata** si possono organizzare percorsi di diffusione espliciti. Le topologie più adatte a questo scopo sono gli *alberi* (dove esiste un unico percorso tra due nodi) e le *mesh* (che invece offrono più percorsi possibili). In particolare è possibile costruire tramite Chord (non lo vediamo) un albero di multicast.

Nell'**overlay non strutturata** invece la propagazione procede tramite i contatti dei nodi, senza un percorso di distribuzione prestabilito per ogni messaggio --> in questo scenario si introduce il gossiping.

---

Similmente alla RW, nel gossiping il nodo **contatta un sottinsieme casuale dei vicini che conosce**, e **i nodi che apprendono l'informazione possono a loro volta diffonderla** --> *il gossiping cerca di ridurre il numero di msg inviati rispetto al flooding, cercando al contempo di essere più veloce rispetto alla RW.*

Il nome richiama "pettegolezzo" non a caso: raccontando un gossip a qualcuno, quello lo potrà raccontare ai propri vicini e così via... si paragona del resto il gossiping anche al concetto di epidemia --> protocolli di gossiping ed epidemici sono della stessa famiglia, ed entrambi non deterministici.    
Un nodo si dice **contaminato** quando ha appreso l'aggiornamento e può contribuire a diffonderlo.

Si collega il gossiping a tre proprietà:
- **Semplicità degli algoritmi**
- **Decentralizzazione**: ogni nodo sceglie i propri contatti casualmente, senza alcuna coordinazione centrale.
- **Scalabilità**: il numero di contatti avviati da ciascun nodo **per ciclo** (la comunicazione in gossip può essere in differita: appena un nodo riceve l'info può decidere di diffonderla dopo un certo tempo) può essere limitato all'aumentare di $N$
- **Robustezza e Resilienza**: l'informazione può raggiungere un nodo attraverso percorsi diversi, per cui se un nodo o link fallisce, l'informazione può comunque arrivare a destinazione (message redundancy).

es. applicativi: Amazon S3 per la diffusione di info utili a evitare nodi irraggiungibili, in Dynamo la failure detection dei nodi, Cassandra per la propagazione di info sul group membership e failure detection....

Due strategie fondamentali per il gossiping: l'**Anti-Entropy** e **Rumor Spreading**.

### Anti-Entropy
L'**Anti-Entropy** parte dallo stato dei nodi e cerca di ridurne le differenze (ridurre in un certo senso cioè il "disordine", da qui il nome!).

Periodicamente, ogni nodo ($P$) sceglie $u.a.r.$ un nodo ($Q$) e ci si scambiano gli aggiornamenti secondo tre modalità, con l'obiettivo di avere lo stesso stato alla fine del processo:
- **Push** $P \Rightarrow Q$: in questo caso se $P$ ha l'aggiornamento, aiuta a diffonderlo mandandolo a $Q$
- **Pull** $P \Leftarrow Q$: in questo caso $P$ chiede gli aggiornamenti, e $Q$ lo aiuta se ha quello che manca a $P$
- **Push-Pull** $P \Leftrightarrow Q$: questa modalità è la più efficiente, perché se uno dei due ha l'aggiornamento, lo trasferisce all'altro.

Il push-pull è il più veloce in quanto all'inizio, quando pochi nodi conoscono la novità, è il *push* a permettere a quei pochi nodi di diffonderla.  
Verso la fine invece, quando quasi tutti la conoscono, il *pull* aiuta gli ultimi nodi in quanto hanno buone probabilità di contattare qualcuno che è già aggiornato.

Nell'immagine seguente, sull'asse x ci sono i **round** (intervalli nei quali ogni nodo avvia uno scambio), mentre sull'asse y i nodi che **ancora non conoscono l'aggiornamento** --> si osserva che la curva che scende più velocemente (e quindi più veloce) è proprio push-pull.

<img src="imgs/7_22.png" width="400">

### Rumor Spreading
Nel **Rumor Spreading** un nodo con un aggiornamento (**contaminato**) sceglie periodicamente $F \ge 1$ (*fanout*) vicini e gli manda l'update per contaminarli a loro volta. Un nodo che contaminato può smettere di diffondere l'aggiornamento con una certa probabilità.

Nell'approccio **feedback-based**, quando un nodo contaminato $P$ contatta un nodo che già conosce l'aggiornamento, smette di propagare quell'aggiornamento con probabilità $p_{stop}$.  
Nell'esempio qui sotto si assume fanout $F = 1$, e si osserva come la frazione dei nodi rimasti incontaminati ($s$, sull'asse $y$) cresce all'aumentare di $p_{stop}$ (sull'asse $x$), arrivando comunque a un $80\%$ di nodi contaminati anche con $p_{stop} = 1$!.

Si può infatti dimostrare che esiste una relazione trascendentale tra $s$ e $p_{stop}$:
$$s = e^{-\left(1+\frac{1}{p_{stop}}\right)(1-s)}$$
che è trascendentale perché l'incognita $s$ compare anche all'esponente.

Per ridurre ancora di più la frazione di nodi incontaminati nel tempo, l'idea è *combinare il rumor spreading con l'anti-entropy*.